# BT5151 Toxic Comment Agent Demo (Colab)

This notebook demonstrates the online LangGraph moderation pipeline for the toxic comment agent:

`run-inference -> assess-severity -> recommend-moderation-action`

It is designed for Google Colab and focuses on the deploy-time moderation flow rather than retraining models.

## 0. Before You Run

Make sure this notebook is opened from the project root in Colab, or update `PROJECT_ROOT` in the next setup cell.

Required files expected in the project folder:
- `select_model_output.json`
- `models/selected_model_metadata.json`
- trained model artefacts under `models/`
- `pipeline/state.py`
- `pipeline/graph.py`

In [10]:
%pip install -q \
  numpy==1.26.4 \
  scipy==1.11.4 \
  scikit-learn==1.3.2 \
  pandas==2.2.2 \
  matplotlib==3.8.4 \
  torch torchvision \
  transformers==4.46.3 \
  sentence-transformers==3.2.1 \
  langgraph \
  gradio==4.44.1


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/BT5151_toxic_comment_agent").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

required_paths = [
    PROJECT_ROOT / 'select_model_output.json',
    PROJECT_ROOT / 'models' / 'selected_model_metadata.json',
    PROJECT_ROOT / 'pipeline' / 'state.py',
    PROJECT_ROOT / 'pipeline' / 'graph.py',
]

for path in required_paths:
    print(path, 'OK' if path.exists() else 'MISSING')

/content/drive/MyDrive/BT5151_toxic_comment_agent/select_model_output.json OK
/content/drive/MyDrive/BT5151_toxic_comment_agent/models/selected_model_metadata.json OK
/content/drive/MyDrive/BT5151_toxic_comment_agent/pipeline/state.py OK
/content/drive/MyDrive/BT5151_toxic_comment_agent/pipeline/graph.py OK


In [4]:
import json
from typing import Any

import gradio as gr

from pipeline.graph import run_pipeline

print('Imports loaded successfully.')

Imports loaded successfully.


## 1. Quick Smoke Test

In [5]:
sample_result = run_pipeline('You are an absolute idiot and nobody wants you here.')
print('Recommended action :', sample_result['action_label'])
print('Severity           :', sample_result['severity_label'])
print('Review priority    :', sample_result['review_priority'])
print('UI explanation     :', sample_result['ui_explanation'])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Recommended action : Remove and Escalate
Severity           : critical
Review priority    : urgent
UI explanation     : This comment was assessed as critical. Remove the comment immediately and escalate the case to the urgent moderation queue.


## 2. UI Helper Function

In [6]:
def analyze_comment_for_ui(comment_text: str):
    clean_text = str(comment_text).strip()
    if not clean_text:
        empty_payload = {'error': 'Please enter a comment before running the moderation pipeline.'}
        return (
            'No action available',
            'No business summary available.',
            'unknown',
            'unknown',
            'unknown',
            'Enter a comment to begin.',
            False,
            empty_payload,
        )

    result = run_pipeline(clean_text)
    return (
        result.get('action_label', 'Unknown'),
        result.get('business_message', 'No business summary available.'),
        result.get('severity_label', 'unknown'),
        result.get('review_priority', 'unknown'),
        result.get('user_notification', 'unknown'),
        result.get('ui_explanation', 'No explanation available.'),
        bool(result.get('human_review_required', False)),
        result,
    )

## 3. Launch Gradio Demo

In [7]:
with gr.Blocks(title='BT5151 Toxic Comment Moderation Agent') as demo:
    gr.Markdown(
        '''
        # BT5151 Toxic Comment Moderation Agent

        This demo runs the LangGraph moderation pipeline:
        `run-inference -> assess-severity -> recommend-moderation-action`
        '''
    )

    with gr.Row():
        with gr.Column(scale=3):
            comment_input = gr.Textbox(
                label='Comment Text',
                placeholder='Enter a comment to classify and moderate...',
                lines=6,
            )
            analyze_button = gr.Button('Run Moderation Pipeline', variant='primary')

            gr.Examples(
                examples=[
                    ['Thank you for your edits, this article is much clearer now.'],
                    ['This is a stupid comment and your argument makes no sense.'],
                    ['You are an absolute idiot and nobody wants you here.'],
                    ['What the hell is going on with this page?'],
                ],
                inputs=[comment_input],
            )

        with gr.Column(scale=2):
            action_label = gr.Textbox(label='Recommended Action', interactive=False)
            business_message = gr.Textbox(label='Business Message', interactive=False, lines=3)
            severity_label = gr.Textbox(label='Severity', interactive=False)
            review_priority = gr.Textbox(label='Review Priority', interactive=False)
            user_notification = gr.Textbox(label='User Notification', interactive=False)
            ui_explanation = gr.Textbox(label='UI Explanation', interactive=False, lines=4)
            human_review_required = gr.Checkbox(label='Human Review Required', interactive=False)

    raw_output = gr.JSON(label='Full Pipeline Output')

    analyze_button.click(
        fn=analyze_comment_for_ui,
        inputs=[comment_input],
        outputs=[
            action_label,
            business_message,
            severity_label,
            review_priority,
            user_notification,
            ui_explanation,
            human_review_required,
            raw_output,
        ],
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f3a2335a3a10540247.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f3a2335a3a10540247.gradio.live


## 4. Suggested Test Cases for the Presentation

Use these in your recorded demo or report:

1. Clean case: `Thank you for your edits, this article is much clearer now.`
2. Borderline / mild abuse: `This is a stupid comment and your argument makes no sense.`
3. Severe toxic case: `You are an absolute idiot and nobody wants you here.`
4. Ambiguous case: `What the hell is going on with this page?`